# G1: đánh giá lại các checkpoint đã train (64 robot × 10 giây)

Phép đánh giá cũ (16 robot × 5 giây) quá nhiễu: cùng một cấu hình mà sai số ở lệnh 1,5 m/s dao
động từ 0,037 đến 0,071 m/s. Notebook này **không train**. Nó đánh giá lại checkpoint cuối
(`model_1999.pt`) của mọi lần train đã chạy trên Kaggle với:

- **64 robot**, mỗi lệnh **11 giây** (1 giây đầu để tăng tốc, đo trong **10 giây** sau đó);
- lệnh thẳng 0,5 / 1,0 / 1,5 m/s, cùng một seed đánh giá cho mọi checkpoint.

Kết quả gồm bảng từng lần train (kèm số cũ để so) và bảng gộp theo phương pháp
(trung bình ± độ lệch chuẩn giữa các lần train).

## Chuẩn bị (panel bên phải)

1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. **Add Input → Your Work → Notebooks**, thêm **các notebook đã train** trước đây, mỗi notebook một
   lần (ví dụ notebook so sánh 4 phương pháp, notebook thử cấu hình A/B, notebook tối ưu). Kaggle
   gắn output của phiên bản đã chạy; nếu hộp thoại cho chọn phiên bản, chọn phiên bản đã train xong.
   Notebook tự tìm mọi `model_*.pt` trong `/kaggle/input`.

Sau đó **Save Version → Save & Run All (Commit)**. Khoảng 3–5 phút mỗi checkpoint, chia cho 2 GPU.
Kết quả: `mpc_g1_eval_results.zip` ở tab **Output**.

## 0. Cấu hình

In [ ]:
import os
import time

NOTEBOOK_START = time.time()

REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"
BRANCH = "mpc-stage1"
# Mọi phương pháp dùng cùng môi trường và cùng mạng; task PPO không dựng thầy MPC khi đánh giá.
EVAL_TASK = "Mpopi-G1-2k-PPO"
EVAL_ENVS = 64
SETTLE_STEPS = 50  # 1 giây đầu (50 Hz) không tính vào vận tốc trung bình.
EVAL_STEPS = SETTLE_STEPS + 500  # Đo trong 10 giây.
SPEEDS = "0.5 1.0 1.5"
EVAL_SEED = 10_000

REPO_DIR = "/tmp/mjlab_MPOPI"  # Không để trong /kaggle/working (thư mục output).
OUT_ROOT = "/kaggle/working/mpc_g1_eval"
EVAL_DIR = f"{OUT_ROOT}/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

In [ ]:
#@title Hàm tiện ích
import subprocess


def run(cmd, cwd=None, check=True):
  """Run a shell command and stream its output."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0 and check:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
  return proc.returncode

## 1. Kiểm tra GPU

In [ ]:
gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu_list)
GPUS = [line.split(":")[0].split()[-1] for line in gpu_list.splitlines() if line.startswith("GPU")]
if not GPUS:
  raise RuntimeError("Không thấy GPU. Settings → Accelerator → GPU T4 x2.")
print(f"Dùng {len(GPUS)} GPU: {GPUS}")

## 2. Lấy code và cài môi trường

Lần đầu mất khoảng 5–10 phút.

In [ ]:
import shutil

if not REPO_DIR.startswith("/tmp/"):
  raise RuntimeError("REPO_DIR phải nằm trong /tmp (ô này xóa rồi clone lại thư mục đó).")
if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
run("uv sync --extra cu128", cwd=REPO_DIR)
UV = "uv run --extra cu128"

## 3. Tìm checkpoint

Mỗi thư mục lần train có dạng `<ngày giờ>_<phương pháp>_s<seed>`. Ngày giờ làm cho tên duy nhất, nên
hai phiên bản cùng tên lần train (ví dụ `PPO_s1` của hai đợt chạy) không bị lẫn. Lần train chưa tới
`model_1999.pt` (bị dừng giữa chừng) được bỏ qua.

In [ ]:
import glob
import re

RUNS = {}  # Tên thư mục lần train -> thông tin.
for ckpt in glob.glob("/kaggle/input/**/train_logs/g1_velocity_2k/*/model_*.pt", recursive=True):
  run_dir = os.path.dirname(ckpt)
  run_id = os.path.basename(run_dir)
  it = int(re.search(r"model_(\d+)\.pt", ckpt).group(1))
  if run_id in RUNS and RUNS[run_id]["iteration"] >= it:
    continue
  name = run_id[20:]  # Bỏ "YYYY-mm-dd_HH-MM-SS_".
  root = os.path.dirname(os.path.dirname(os.path.dirname(run_dir)))  # Thư mục chứa train_logs.
  RUNS[run_id] = {
    "checkpoint": ckpt, "iteration": it, "name": name,
    "method": re.sub(r"_s\d+$", "", name),
    "old_eval": f"{root}/eval/{name}.json",
    "source": os.path.relpath(root, "/kaggle/input"),
  }
skipped = sorted(r for r, v in RUNS.items() if v["iteration"] < 1999)
for r in skipped:
  print(f"Bỏ qua {r}: checkpoint cuối là vòng {RUNS.pop(r)['iteration']}")
print(f"\n{len(RUNS)} lần train đầy đủ:")
for r, v in sorted(RUNS.items()):
  print(f"  {v['method']:>20}  {r}  ({v['source']})")
if not RUNS:
  raise RuntimeError("Không thấy checkpoint: hãy Add Input các notebook đã train (xem đầu trang).")

## 4. Đánh giá

Mỗi GPU lần lượt lấy một checkpoint từ hàng đợi. Checkpoint đã có kết quả thì bỏ qua.

In [ ]:
import queue
import threading

jobs = queue.Queue()
for r in sorted(RUNS):
  if not os.path.exists(f"{EVAL_DIR}/{r}.json"):
    jobs.put(r)
print(f"{jobs.qsize()} checkpoint cần đánh giá", flush=True)


def lane(gpu):
  while True:
    try:
      r = jobs.get_nowait()
    except queue.Empty:
      return
    t0 = time.time()
    cmd = (
      f"{UV} mpopi-eval --task {EVAL_TASK} --device cuda:0 --controllers policy "
      f"--num-envs {EVAL_ENVS} --steps {EVAL_STEPS} --settle-steps {SETTLE_STEPS} "
      f"--speeds {SPEEDS} --seed {EVAL_SEED} --checkpoint '{RUNS[r]['checkpoint']}' "
      f"--out '{EVAL_DIR}/{r}.json'"
    )
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu}
    with open(f"{EVAL_DIR}/{r}.log", "w") as f:
      rc = subprocess.run(cmd, shell=True, cwd=REPO_DIR, env=env, stdout=f,
                          stderr=subprocess.STDOUT).returncode
    status = "xong" if rc == 0 else f"LỖI (mã {rc}), xem {EVAL_DIR}/{r}.log"
    print(f"[GPU {gpu}] {r}: {status} sau {time.time() - t0:.0f} s", flush=True)


threads = [threading.Thread(target=lane, args=(g,)) for g in GPUS]
for t in threads:
  t.start()
for t in threads:
  t.join()
print(f"Xong sau {(time.time() - NOTEBOOK_START) / 60:.0f} phút từ đầu notebook.")

## 5. Kết quả

**Bảng 1:** từng lần train. Cột *cũ* là kết quả đánh giá 16 robot × 5 giây lúc train (nếu có).
**Bảng 2:** gộp theo phương pháp, trung bình ± độ lệch chuẩn giữa các lần train.
Đạt tiêu chí bám 1,5 m/s khi vận tốc ≥ 1,2 m/s và ≤ 0,25 lần ngã/robot.

In [ ]:
import json

import numpy as np


def policy_rows(path):
  if not os.path.exists(path):
    return {}
  rows = json.load(open(path))["results"]
  return {r["command"]: r for r in rows if r["controller"] == "policy"}


rows = []
for r, v in sorted(RUNS.items(), key=lambda kv: (kv[1]["method"], kv[0])):
  new = policy_rows(f"{EVAL_DIR}/{r}.json")
  if not new:
    continue
  old = policy_rows(v["old_eval"])
  rows.append({
    "run": r, "method": v["method"],
    **{f"v{c}": new[c]["speed"] for c in new},
    **{f"e{c}": new[c]["abs_error"] for c in new},
    **{f"f{c}": new[c]["falls_per_env"] for c in new},
    "old_e1.5": old[1.5]["abs_error"] if 1.5 in old else np.nan,
  })

lines = [f"{'lần train':>48} {'v 0,5':>6} {'v 1,0':>6} {'v 1,5':>6} {'|e| 0,5':>8} "
         f"{'|e| 1,0':>8} {'|e| 1,5':>8} {'cũ 1,5':>7} {'ngã 1,5':>8}"]
for x in rows:
  lines.append(f"{x['run'][-48:]:>48} {x['v0.5']:>6.2f} {x['v1.0']:>6.2f} {x['v1.5']:>6.2f} "
               f"{x['e0.5']:>8.3f} {x['e1.0']:>8.3f} {x['e1.5']:>8.3f} {x['old_e1.5']:>7.3f} "
               f"{x['f1.5']:>8.2f}")
table1 = "\n".join(lines)

methods = sorted({x["method"] for x in rows})
fmt = lambda a: f"{np.mean(a):.3f} ± {np.std(a, ddof=1) if len(a) > 1 else 0:.3f}"
lines = [f"{'phương pháp':>20} {'số lần':>6} {'vận tốc 1,5':>15} {'|e| 1,5':>15} "
         f"{'|e| 1,0':>15} {'|e| 0,5':>15} {'đạt':>5}"]
for m in methods:
  xs = [x for x in rows if x["method"] == m]
  get = lambda k: [x[k] for x in xs]
  ok = sum(x["v1.5"] >= 1.2 and x["f1.5"] <= 0.25 for x in xs)
  lines.append(f"{m:>20} {len(xs):>6} {fmt(get('v1.5')):>15} {fmt(get('e1.5')):>15} "
               f"{fmt(get('e1.0')):>15} {fmt(get('e0.5')):>15} {ok:>2}/{len(xs)}")
table2 = "\n".join(lines)

report = (f"Đánh giá {EVAL_ENVS} robot × {(EVAL_STEPS - SETTLE_STEPS) / 50:.0f} giây "
          f"(seed {EVAL_SEED})\n\nBảng 1: từng lần train\n{table1}\n\n"
          f"Bảng 2: theo phương pháp (trung bình ± độ lệch chuẩn)\n{table2}\n")
print(report)
open(f"{OUT_ROOT}/ket_qua_danh_gia_lai.txt", "w").write(report)

## 6. Đóng gói

Kết quả nén ở `/kaggle/working/mpc_g1_eval_results.zip`, tải ở tab **Output**.

In [ ]:
import zipfile

src = "/kaggle/working/mpc_g1_eval_results.zip"
with zipfile.ZipFile(src, "w", zipfile.ZIP_DEFLATED) as zf:
  for root, _, files in os.walk(OUT_ROOT):
    for fname in files:
      path = os.path.join(root, fname)
      zf.write(path, os.path.relpath(path, OUT_ROOT))
print(f"Xong. Kích thước zip: {os.path.getsize(src) / 1e6:.2f} MB")